# Signal Detection Plots Viewer

This notebook displays the debug plots generated during the realistic replay.
Plots are created only when a new signal (maximum or minimum) is detected.
Each plot shows the time series window and detected extrema at the moment of signal detection.

In [ ]:
# ========================================
# CONFIGURATION - Modify these variables
# ========================================

# Data configuration
data_folder = "2025_05_30_01_mc"
strategy_folder = "extrema"
strategy_name = "var12ge0_83x0-var15ge0_73__w7md5p0_05"
series_id = "1"

# Optional: Filter by iteration range (set to None to view all)
start_iter = None  # e.g., 0 or None for all
end_iter = None    # e.g., 10 or None for all

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
from PIL import Image
import re

# Get project root
project_root = Path.cwd().parent
print(f"Project root: {project_root}")

In [ ]:
# Build path to debug directory using configuration
base_path = project_root / "data" / data_folder / strategy_folder / strategy_name / "signal" / "debug"
debug_dir = base_path / series_id

print(f"Looking for debug plots in: {debug_dir}")

if not debug_dir.exists():
    print(f"❌ Debug directory not found!")
    print(f"\nSearching for available series in: {base_path}")
    if base_path.exists():
        series_dirs = [d for d in base_path.iterdir() if d.is_dir()]
        if series_dirs:
            print(f"Found {len(series_dirs)} series with debug plots:")
            for series_dir in sorted(series_dirs):
                plot_count = len(list(series_dir.glob("*_th.png")))
                print(f"  - {series_dir.name}: {plot_count} plots")
        else:
            print("No series directories found.")
    else:
        print("Base path does not exist. Please run the categorization task first.")
else:
    print(f"✅ Debug directory found!")

In [ ]:
# Get all debug plots and sort them by iteration number (Current Date idx)
plot_files = list(debug_dir.glob("*_th.png"))

# Extract iteration number and type suffix from filename and sort
def get_sort_key(filepath):
    """
    Extract sort key from filename.
    New format: '{idx}_{type}{count}_th.png' (e.g., '500_e1_th.png', '500_s2_th.png')
    Old format: '{idx}_th.png' (e.g., '500_th.png')
    
    Sort order: by idx first, then suppressed (s) before events (e), then by count
    """
    # Try new format first: {idx}_{type}{count}_th.png
    match = re.search(r'(\d+)_([es])(\d+)_th\.png$', filepath.name)
    if match:
        idx = int(match.group(1))
        event_type = match.group(2)  # 'e' for event, 's' for suppressed
        count = int(match.group(3))
        # Sort: by idx, then 's' before 'e' (0 for suppressed, 1 for event), then by count
        type_rank = 0 if event_type == 's' else 1
        return (idx, type_rank, count)
    
    # Try old format: {idx}_th.png
    match = re.search(r'(\d+)_th\.png$', filepath.name)
    if match:
        return (int(match.group(1)), 2, 0)  # Put old format after new ones
    
    return (-1, 0, 0)

def get_iteration_number(filepath):
    """Extract iteration number from filename for filtering purposes."""
    key = get_sort_key(filepath)
    return key[0]

plot_files_sorted = sorted(plot_files, key=get_sort_key)

# Apply iteration range filter if specified
if start_iter is not None or end_iter is not None:
    _start = start_iter if start_iter is not None else -float('inf')
    _end = end_iter if end_iter is not None else float('inf')
    plot_files_sorted = [p for p in plot_files_sorted if _start <= get_iteration_number(p) <= _end]
    print(f"Filtered to iteration range: {start_iter} to {end_iter}")

print(f"Found {len(plot_files_sorted)} debug plots")
if plot_files_sorted:
    first_iter = get_iteration_number(plot_files_sorted[0])
    last_iter = get_iteration_number(plot_files_sorted[-1])
    print(f"Iteration range: {first_iter} to {last_iter}")

In [ ]:
# Display all plots sequentially
print(f"📊 Displaying {len(plot_files_sorted)} signal detection plots for series '{series_id}'\n")
print("=" * 80)

for plot_file in plot_files_sorted:
    iteration = get_iteration_number(plot_file)
    
    # Display the image
    img = Image.open(plot_file)
    display(img)
    
    # Add separator
    display(HTML("<hr style='margin: 20px 0; border: none; border-top: 1px solid #ecf0f1;'>"))

print("✅ All plots displayed successfully!")